In [1]:
from pathlib import Path

print("Current folder:", Path.cwd())

pdf_files = list(Path("DATA").glob("*.pdf"))

print("Found PDFs:", len(pdf_files))

for pdf in pdf_files[:5]:
    print(pdf)

Current folder: d:\finance_assistant\backend\RAG
Found PDFs: 25
DATA\AAPL2021.pdf
DATA\AAPL2022.pdf
DATA\AAPL2023.pdf
DATA\AAPL2024.pdf
DATA\AAPL2025.pdf


In [3]:
import fitz
import pytesseract
from PIL import Image
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import io

# Update if your installation path is different
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

PDF_FOLDER = Path("DATA")

pages_and_texts = []

pdf_files = list(PDF_FOLDER.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDFs")

for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):

    try:
        doc = fitz.open(pdf_file)

        use_ocr = pdf_file.name.upper().startswith("AAPL")

        for page_number, page in enumerate(doc):

            if use_ocr:

                # OCR extraction
                pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))

                img = Image.open(io.BytesIO(pix.tobytes("png")))

                text = pytesseract.image_to_string(img)

            else:

                # Normal PyMuPDF extraction
                text = page.get_text()

            text = text.replace("\n", " ").strip()

            pages_and_texts.append({
                "source": pdf_file.name,
                "page_number": page_number + 1,
                "char_count": len(text),
                "word_count": len(text.split()),
                "token_estimate": round(len(text) / 4),
                "text": text
            })

        doc.close()

    except Exception as e:
        print(f"Error processing {pdf_file.name}: {e}")

df = pd.DataFrame(pages_and_texts)

print("\nTotal Pages Extracted:", len(df))

df.to_csv("extracted_pages.csv", index=False)

print("Saved -> extracted_pages.csv")

ModuleNotFoundError: No module named 'fitz'

In [ ]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)

df.head()


,source,page_number,char_count,word_count,token_estimate,text
0,AAPL2021.pdf,1,1805,286,451,"6/15/26, 12:34 PM aapl-20210925 UNITED STATES..."
1,AAPL2021.pdf,2,3229,495,807,"6/15/26, 12:34 PM aapl-20210925 Indicate by ch..."
2,AAPL2021.pdf,3,1439,205,360,"6/15/26, 12:34 PM aapl-20210925 Apple Inc. ..."
3,AAPL2021.pdf,4,4071,614,1018,"6/15/26, 12:34 PM aapl-20210925 This Annual R..."
4,AAPL2021.pdf,5,3832,510,958,"6/15/26, 12:34 PM aapl-20210925 Services Adv..."


In [ ]:
# Get stats
df.describe().round(2)

,page_number,char_count,word_count,token_estimate
count,5658.00,5658.00,5658.00,5658.00
mean,145.92,3677.94,542.53,919.48
std,104.23,2071.68,306.66,517.92
min,1.00,0.00,0.00,0.00
25%,57.00,2260.25,337.00,565.00
50%,123.00,3299.00,485.00,824.50
75%,228.00,4672.00,697.00,1168.00
max,420.00,14154.00,2179.00,3538.00


In [ ]:
df = df[df["text"].notna() & (df["text"].str.strip() != "")]